# Homework 3 - Chapter 4：关联规则挖掘

本 Notebook 完成作业 §6.1 基础作业与 §6.2 拓展练习。主题包括支持度、置信度、提升度、Apriori、关联规则、参数敏感性实验，以及 FP-Growth 思想总结。

## §6.1 基础作业

### 1. 概念理解题

#### （1）支持度、置信度、提升度分别回答什么问题？

**支持度（support）**回答：某个项集或规则涉及的商品组合在全部交易中出现得有多频繁。它衡量的是“这件事是否足够常见”。例如 support({A,B}) = 0.4，表示 40% 的交易同时包含 A 和 B。

**置信度（confidence）**回答：在已经购买了前件 X 的交易中，有多少比例也购买了后件 Y。它衡量的是“看到 X 后，Y 出现的条件概率有多高”。例如 confidence({A}⇒{B}) = 0.8，表示买 A 的人中有 80% 也买 B。

**提升度（lift）**回答：规则 X⇒Y 是否真的比随机情况更有信息量。它比较的是“买了 X 以后买 Y 的概率”相对于“总体上买 Y 的概率”提高了多少。

- lift > 1：X 和 Y 正相关，X 的出现会提高 Y 的出现概率。
- lift = 1：X 和 Y 基本独立，规则没有额外信息。
- lift < 1：X 和 Y 负相关，X 的出现反而降低 Y 的出现概率。

#### 为什么仅看置信度不够？

仅看置信度容易被“本来就很热门的商品”误导。例如，如果 90% 的用户都会买矿泉水，那么很多规则 `{X}⇒{矿泉水}` 的置信度都会很高，但这不代表 X 和矿泉水之间真的有特殊关系。提升度会除以 Y 本身的支持度，从而判断该规则是否超过了商品 Y 的基础流行程度。因此，提升度可以过滤掉很多“看似高置信度、实际没价值”的规则。

#### （2）非电商/医疗/金融/教育应用场景

场景：健身房会员行为分析。

希望挖掘到的关联规则示例：

`{参加力量训练课程, 使用蛋白粉补给区} ⇒ {预约体测服务}`

潜在业务价值：如果数据显示参加力量训练并使用补给区的会员更可能预约体测，那么健身房可以在这些会员完成训练后推送体测服务优惠，或者推荐私人教练进行阶段性训练评估。这类规则可以帮助健身房提高增值服务转化率，也能帮助用户更系统地管理训练效果。

### 2. 手算小练习

给定事务数据库：

| 交易 | 项集 |
|---|---|
| T1 | {A, B, C} |
| T2 | {A, C} |
| T3 | {B, C} |
| T4 | {A, B} |
| T5 | {A, B, C} |

共 5 笔交易。

#### （1）计算二项集支持度

- {A, B} 出现在 T1、T4、T5，共 3 次，所以 support({A,B}) = 3/5 = 0.6。
- {A, C} 出现在 T1、T2、T5，共 3 次，所以 support({A,C}) = 3/5 = 0.6。
- {B, C} 出现在 T1、T3、T5，共 3 次，所以 support({B,C}) = 3/5 = 0.6。

#### （2）计算规则置信度和提升度

先计算单项支持度：

- support(A) = 4/5 = 0.8，因为 A 出现在 T1、T2、T4、T5。
- support(B) = 4/5 = 0.8，因为 B 出现在 T1、T3、T4、T5。
- support(C) = 4/5 = 0.8，因为 C 出现在 T1、T2、T3、T5。

规则 `{A}⇒{C}`：

- confidence({A}⇒{C}) = support({A,C}) / support(A) = 0.6 / 0.8 = 0.75。
- lift({A}⇒{C}) = confidence({A}⇒{C}) / support(C) = 0.75 / 0.8 = 0.9375。

规则 `{B}⇒{C}`：

- confidence({B}⇒{C}) = support({B,C}) / support(B) = 0.6 / 0.8 = 0.75。
- lift({B}⇒{C}) = confidence({B}⇒{C}) / support(C) = 0.75 / 0.8 = 0.9375。

#### （3）哪条规则更值得关注？

从计算结果看，`{A}⇒{C}` 和 `{B}⇒{C}` 的支持度、置信度、提升度完全相同，因此仅根据当前数据无法判断哪一条更值得关注。

更重要的是，两条规则的 lift 都是 0.9375，小于 1，说明 A 或 B 的出现并没有提高 C 的出现概率，反而略低于 C 的整体出现概率。因此这两条规则虽然置信度看起来有 0.75，但并不是很有业务价值。这也说明仅看置信度是不够的。

### 3. 代码实践：使用 mlxtend 在自定义购物篮数据上挖掘关联规则

下面构造 12 条交易记录，并使用 `TransactionEncoder + apriori + association_rules` 挖掘关联规则。

In [ ]:
# 如果本地环境没有安装 mlxtend，请先运行这一行
# %pip install mlxtend -q

import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
transactions = [
    ['牛奶', '面包', '鸡蛋'],
    ['牛奶', '面包'],
    ['牛奶', '鸡蛋', '黄油'],
    ['面包', '鸡蛋', '黄油'],
    ['牛奶', '面包', '鸡蛋', '黄油'],
    ['咖啡', '面包'],
    ['咖啡', '牛奶', '面包'],
    ['尿布', '啤酒', '薯片'],
    ['尿布', '啤酒', '牛奶'],
    ['面包', '黄油'],
    ['牛奶', '面包', '香蕉'],
    ['牛奶', '香蕉', '鸡蛋'],
]

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

df.head()

In [ ]:
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False])

In [ ]:
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)

rules_display = rules[[
    'antecedents', 'consequents', 'support', 'confidence', 'lift'
]].copy()

rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(list(x)))
rules_display.sort_values(['confidence', 'lift'], ascending=False)

#### 规则解释

1. `{鸡蛋} ⇒ {牛奶}`：在这个样本数据中，买鸡蛋的交易中有较高比例也买了牛奶，且 lift > 1，说明鸡蛋和牛奶之间存在一定正相关关系。业务上可以把牛奶放在鸡蛋附近，或者在用户购买鸡蛋时推荐牛奶。

2. `{牛奶} ⇒ {面包}` 与 `{面包} ⇒ {牛奶}`：这两条规则的置信度达到 0.6 以上，说明牛奶和面包经常一起出现。但它们的 lift 低于 1，说明这种关系可能主要来自牛奶、面包本身都比较常见，并不一定代表强关联。因此业务上可以参考，但不应作为重点推荐规则。

## §6.2 拓展练习

### 1. 参数敏感性实验：min_support 与 min_confidence

使用上面构造的购物篮数据进行实验。

In [ ]:
def count_rules_by_thresholds(df, min_support, min_confidence):
    freq = apriori(df, min_support=min_support, use_colnames=True)
    if freq.empty:
        return 0
    rules_tmp = association_rules(freq, metric='confidence', min_threshold=min_confidence)
    return len(rules_tmp)

support_values = [0.1, 0.2, 0.3, 0.4, 0.5]
confidence_fixed = 0.6
support_rule_counts = [count_rules_by_thresholds(df, s, confidence_fixed) for s in support_values]

support_experiment = pd.DataFrame({
    'min_support': support_values,
    'rule_count': support_rule_counts
})
support_experiment

In [ ]:
confidence_values = [round(x / 10, 1) for x in range(3, 10)]
support_fixed = 0.3
confidence_rule_counts = [count_rules_by_thresholds(df, support_fixed, c) for c in confidence_values]

confidence_experiment = pd.DataFrame({
    'min_confidence': confidence_values,
    'rule_count': confidence_rule_counts
})
confidence_experiment

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(support_experiment['min_support'], support_experiment['rule_count'], marker='o')
plt.xlabel('min_support')
plt.ylabel('Number of Rules')
plt.title('Rule Count vs min_support, min_confidence=0.6')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(confidence_experiment['min_confidence'], confidence_experiment['rule_count'], marker='o')
plt.xlabel('min_confidence')
plt.ylabel('Number of Rules')
plt.title('Rule Count vs min_confidence, min_support=0.3')
plt.grid(True)
plt.show()

#### 参数敏感性讨论

从实验可以看出，`min_support` 越高，频繁项集数量通常越少，能生成的规则数量也会下降。原因是支持度阈值提高后，很多出现次数较少的组合会被直接过滤掉，后续也就无法生成规则。

`min_confidence` 越高，规则数量也会减少。因为置信度阈值越严格，只有条件概率足够高的规则才能被保留。

如果我是业务方，我更倾向于先把 `min_support` 设在 0.2–0.3 左右，把 `min_confidence` 设在 0.6–0.8 左右。这样既不会因为阈值太高而漏掉有潜力的长尾规则，也不会因为阈值太低产生大量噪声规则。实际业务中还应该结合 lift、规则可解释性、利润率、库存情况等指标进一步筛选。

### 2. 从频繁项集到推荐系统的简单构想

假设用户当前购物车中的商品集合为 C，规则库中已经有若干形如 `{X}⇒{Y}` 的高置信度、高提升度规则。一个简单推荐策略可以设计如下：

1. 遍历规则库，筛选出前件 X 是当前购物车 C 的子集的规则，即满足 `X ⊆ C`。
2. 对每条满足条件的规则，取出后件 Y。如果 Y 中的商品已经在购物车中，则不再推荐。
3. 对候选推荐商品按照综合分数排序，例如：`score = confidence × lift × support`，也可以加入利润率、库存、用户历史偏好等业务权重。
4. 只推荐 Top-K 个商品，例如最多推荐 3 个，避免页面过载。
5. 过滤“常识性垃圾推荐”。例如，若某商品本身支持度极高但 lift 接近 1，说明它只是热门，并非真正由当前购物车触发的个性化推荐，应降低权重或剔除。
6. 合并重复推荐。如果多个规则推荐同一个商品，只保留最高分或累积分数最高的版本。

这种策略的核心是：不要简单推荐最热门商品，而是推荐“因为当前购物车而变得更值得推荐”的商品。

### 3. FP-Growth 思想阅读与总结

FP-Growth 和 Apriori 都用于挖掘频繁项集，但核心思路不同。Apriori 依赖“频繁项集的子集也必须频繁”这一性质，逐层生成候选项集，再不断扫描数据库统计支持度。它的优点是思想直观、容易实现，适合小规模数据和教学场景；缺点是当商品种类多、事务数量大时，会产生大量候选集，计算成本较高。

FP-Growth 则先扫描数据库，找出频繁 1-项集，并按频率排序；之后把每条交易按频率顺序插入 FP 树，把共享前缀压缩到同一条路径中。挖掘时，它通过条件模式基和条件 FP 树递归寻找频繁模式，不需要像 Apriori 那样显式生成大规模候选集。因此 FP-Growth 通常在大数据集上更高效，尤其适合存在大量共享前缀的交易数据。但它的缺点是实现和理解更复杂，树结构在数据特别稀疏时压缩效果可能有限。总体看，Apriori 更适合入门和小数据，FP-Growth 更适合较大规模的高效频繁模式挖掘。